In [28]:
import os
import time
import pickle
import streamlit as st
import langchain
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_openai import OpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

In [5]:
import langchain
print(langchain.__version__)
print(langchain.__file__)

1.3.14
C:\Users\addwi\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain\__init__.py


In [26]:
pip install langchain-nvidia-ai-endpoints

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install streamlit

   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.3 MB 2.6 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/10.3 MB 2.3 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/10.3 MB 2.1 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/10.3 MB 1.7 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/10.3 MB 1.7 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/10.3 MB 1.7 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/10.3 MB 1.7 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/10.3 MB 1.7 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/10.3 MB 1.7 MB/s eta 0:00:05
   ------------- -------------------------- 3.4/10.3 MB 1.7 MB/s eta 0:00:05
   --------------- ------------------------ 3.9/10.3 MB 1.7 MB/s eta 0:00:04
   ------------------ --------------------- 4.7/10.3 MB 1.8 MB/s eta 0:00:04
   ---


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
os.environ["NVIDIA_API_KEY"] = "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"

In [15]:
# Initialise LLM with required params
llm = ChatNVIDIA(temperature=0.9, max_tokens=500) 

C:\Users\addwi\AppData\Local\Temp\ipykernel_9944\1160162713.py:2: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(temperature=0.9, max_tokens=500)


(1) Load data

In [16]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

(2) Split data to create chunks

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)   

In [18]:
len(docs) 

19

In [19]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nLoans @ 9.99% in Just 5 Mins\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol AD Lite\n\nMoneycontrol AD Lite\n\nMoneycontrol PRO\n\nMoneycontrol Super PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTren

3) Create embeddings for these chunks and save them to FAISS index

In [22]:
embeddings = NVIDIAEmbeddings()

vectorindex_openai = FAISS.from_documents(docs, embeddings)

In [23]:
# Storing vector index create in local
file_path="vector_index.pkl"
with open(file_path, "wb") as f:
    pickle.dump(vectorindex_openai, f)

In [24]:
if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        vectorIndex = pickle.load(f)

4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

In [25]:
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vectorIndex.as_retriever())
chain

RetrievalQAWithSourcesChain(verbose=False, combine_documents_chain=MapReduceDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following portion of a long document to see if any of the text is relevant to answer the question.\nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=ChatNVIDIA(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-nvidia-ai-endpoints': '1.4.3'}}, profile={'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, base_url='https://integrate.api.nvidia.com/v1', model='nvidia/nemotron-3-nano-30b-a3b', temperature=0.9, max_tok

In [31]:
query = "what is the price of Tiago iCNG?"
# query = "what are the main features of Punch iCNG?"

langchain.debug = True

# Run the chain and store the output
result = chain({"question": query}, return_only_outputs=True)

# Print only the answer
print("Answer:")
print(result["answer"])

# Print the source
print("\nSource:")
print(result["sources"])

Answer:
FINAL ANSWER: The Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh.  



Source:
https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
